# Baseline Evaluation — All Experiments (N=2000)
Loads saved scribbles from GDrive, runs full N=2000 evaluation for all methods across all experiments.

In [ ]:
import os, sys, json
import numpy as np
import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path
from IPython.display import display
from tqdm.notebook import tqdm

# ── sys.path: add src/ relative to this notebook ─────────────────────────────
_NB_DIR  = Path().resolve()
_REPO    = _NB_DIR.parent if (_NB_DIR.parent / 'src').exists() else _NB_DIR
_SRC_DIR = str(_REPO / 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from models     import load_models
from clip_utils import load_clip_model, encode_images_clip
from metrics    import compute_mmd
from scripts.run_mlgd_f import compute_clip_softmax  # noqa: used in eval loop

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


In [ ]:
## 2. Config

# Path to experiments/ directory — relative to repo root
REPO_EXPERIMENTS   = str(_REPO / 'experiments')
N_EVAL             = 2000
N_PREVIEW          = 5

SEED               = 42
CONTROLNET_SCALE   = 0.5
NEUTRAL_PROMPT     = 'a superrealistic professional photograph of'
MAN_PROMPT         = 'a superrealistic portrait photograph of a man, studio lighting'
WOMAN_PROMPT       = 'a superrealistic portrait photograph of a woman, studio lighting'

EXPERIMENT_CONFIGS = {
    'SkewedTarget': dict(
        mode   = 'binary',
        groups = [
            dict(label='Man',   prompt='a superrealistic portrait photograph of a man, studio lighting',   frac=0.25, color='royalblue'),
            dict(label='Woman', prompt='a superrealistic portrait photograph of a woman, studio lighting', frac=0.75, color='crimson'),
        ],
    ),
    'BalancedTarget': dict(
        mode   = 'binary',
        groups = [
            dict(label='Man',   prompt='a superrealistic portrait photograph of a man, studio lighting',   frac=0.5, color='royalblue'),
            dict(label='Woman', prompt='a superrealistic portrait photograph of a woman, studio lighting', frac=0.5, color='crimson'),
        ],
    ),
    'GenderInterpolation': dict(
        mode   = 'multiclass',
        groups = [
            dict(label='Woman',                  prompt='superrealistic portrait photograph of a woman, extremely feminine features, studio lighting',                                                               frac=0.25, color='crimson'),
            dict(label='Woman w/ masc features', prompt='a superrealistic portrait photograph of a woman with masculine features, heavy brow ridge, studio lighting',                                               frac=0.25, color='orchid'),
            dict(label='Man w/ fem features',    prompt='a superrealistic portrait photograph of a man with extremely feminine features, soft delicate face, high cheekbones, studio lighting',                    frac=0.25, color='slategray'),
            dict(label='Man',                    prompt='a superrealistic portrait photograph of a man, extremely masculine features, studio lighting',                                                             frac=0.25, color='steelblue'),
        ],
    ),
    'AgeInterpolation': dict(
    mode      = 'age',
    age_min   = 40,
    age_max   = 79,
    age_step  = 1,
),
}
SCRIBBLE_FILES = {
    'source':      'scribble_source.png',
    'avg':         'scribble_avg.png',
    'sdedit':      'scribble_sdedit.png',
    'sdedit_best': 'scribble_sdedit_best.png',
    'lgd_cm':      'scribble_mlgdd.png',
}

METHOD_NAMES = ['source', 'avg', 'sdedit', 'sdedit_best', 'lgd_cm']
METHOD_COLORS = {
    'source':      '#888888',
    'avg':         '#4C72B0',
    'sdedit':      '#4C72B0',
    'sdedit_best': '#4C72B0',
    'lgd_cm':      '#2ca02c',
}

print('Config OK')

In [ ]:
## 3. Load Models
architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

In [ ]:
## 4. Helpers

def gen_images(scribble_pil, prompt, n, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = torch.Generator(device=sprinter.device).manual_seed(seed) if seed is not None else None
    imgs = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            imgs.extend(sprinter(
                prompt=[prompt]*bs, image=[scribble_pil]*bs,
                num_inference_steps=2, guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil', generator=generator,
            ).images)
    sprinter.vae.to(dtype=torch.float32)
    return imgs

def clip_embed(images):
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in images]).to(device)
    clip_model.to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    clip_model.to('cpu')
    return embs

def binomial_ci(n_success, n_total, z=1.96):
    p  = n_success / n_total
    se = np.sqrt(p * (1-p) / n_total)
    return p, p - z*se, p + z*se

def show_image_row(images, title, n=N_PREVIEW):
    n_show = min(n, len(images))
    fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
    if n_show == 1: axes = [axes]
    for ax, img in zip(axes, images[:n_show]):
        ax.imshow(img); ax.axis('off')
    fig.suptitle(title, fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()


def load_scribble(base_dir, name):
    p = Path(base_dir) / SCRIBBLE_FILES[name]
    if p.exists():
        return Image.open(p)
    raise FileNotFoundError(f'Scribble not found: {p}')  # ← eject if missing

print('Helpers ready.')

In [ ]:
## 5. Main Evaluation Loop

all_results = {}

for exp_name, cfg in EXPERIMENT_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {exp_name}')
    print(f'{"="*60}')

    # ── Load scribbles ──
    base_dir = Path(REPO_EXPERIMENTS) / exp_name
    scribbles = {}

    try:
      scribbles = {name: load_scribble(base_dir, name) for name in METHOD_NAMES}
    except FileNotFoundError as e:
      raise RuntimeError(f'STOPPING — {exp_name}: {e}')


    # ── Show all scribbles side by side ──
    n = len(scribbles)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (name, img) in zip(axes, scribbles.items()):
        ax.imshow(img, cmap='gray'); ax.set_title(name); ax.axis('off')
    plt.suptitle(f'{exp_name} — All Scribbles', fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()

    # ── Show each scribble individually ──
    for name, scr in scribbles.items():
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(scr, cmap='gray'); ax.set_title(name); ax.axis('off')
        plt.tight_layout(); display(fig); plt.close()

    source_scribble = scribbles['source']
    mode = cfg['mode']

    # ── Build target distribution (N=2000) ──
    print(f'\nBuilding target distribution (N={N_EVAL})...')
    target_imgs_by_group = {}

    if mode in ('binary', 'multiclass'):
        all_target_imgs = []
        for i, g in enumerate(cfg['groups']):
            n_i = max(1, int(N_EVAL * g['frac']))
            print(f"  [{g['label']}] n={n_i}...")
            imgs = gen_images(source_scribble, g['prompt'], n_i, seed=SEED + i*1000)
            all_target_imgs.extend(imgs)
            target_imgs_by_group[g['label']] = imgs
        target_clip = clip_embed(all_target_imgs)

    elif mode == 'age':
        ages     = list(range(cfg['age_min'], cfg['age_max']+1, cfg['age_step']))
        n_per_age   = max(1, N_EVAL // len(ages))   # 2000 // 40 = 50

        age_embs = {}
        clip_model.to(device)
        for age in tqdm(ages, desc='Age target'):
            prompt = (f'a superrealistic portrait photograph of a {age}-year-old man, '
                      'studio lighting, sharp focus, photographic')
            imgs = gen_images(source_scribble, prompt,n_per_age, seed=SEED + age*7)
            tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in imgs]).to(device)
            with torch.no_grad():
                age_embs[age] = encode_images_clip(tensors, clip_model, clip_processor).cpu()
            target_imgs_by_group[str(age)] = imgs
        clip_model.to('cpu')
        target_clip = torch.cat([age_embs[a] for a in ages], dim=0).to(device)

    print(f'Target CLIP: {target_clip.shape}')

    # ── Show target samples per group ──
    sample_groups = list(target_imgs_by_group.items())
    if mode == 'age':
        # show only a few anchor ages
        anchor_ages = [str(ages[0]), str(ages[len(ages)//4]), str(ages[len(ages)//2]),
                       str(ages[3*len(ages)//4]), str(ages[-1])]
        sample_groups = [(a, target_imgs_by_group[a]) for a in anchor_ages if a in target_imgs_by_group]

    for label, imgs in sample_groups:
        show_image_row(imgs, f'Target: {label}', n=N_PREVIEW)

    # ── Evaluate each method ──
    print(f'\nEvaluating methods...')
    exp_results = {}

    for method_name, scribble in scribbles.items():
        print(f'  [{method_name}] generating {N_EVAL} images...')
        imgs = gen_images(scribble, NEUTRAL_PROMPT, N_EVAL, seed=SEED)
        embs = clip_embed(imgs)
        mmd  = compute_mmd(embs, target_clip).item()

        result = {'mmd': mmd}

        # Gender classification for binary/multiclass
        if mode in ('binary', 'multiclass'):
            sr, _ = compute_clip_softmax(
                imgs, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)
            n_male = sum(1 for r in sr if r['label'] == 'male')
            p, lo, hi = binomial_ci(n_male, N_EVAL)
            result.update(dict(p_male=p, ci_lo=lo, ci_hi=hi,
                               n_male=n_male, n_female=N_EVAL-n_male))
            print(f'    MMD={mmd:.5f}  p(male)={p:.3f}  95%CI=[{lo:.3f},{hi:.3f}]')
        else:
            print(f'    MMD={mmd:.5f}')

        exp_results[method_name] = result

        # Show sample images from this method
        show_image_row(imgs, f'{exp_name} — {method_name} (MMD={mmd:.4f})', n=N_PREVIEW)

    # ── Compute improvement vs source ──
    source_mmd = exp_results['source']['mmd']
    for name, r in exp_results.items():
        r['mmd_improvement']     = source_mmd - r['mmd']
        r['mmd_improvement_pct'] = (source_mmd - r['mmd']) / source_mmd * 100

    all_results[exp_name] = exp_results

    # ── Results table ──
    rows = {}
    for name, r in exp_results.items():
        row = {
            'MMD':              f"{r['mmd']:.5f}",
            'Δ MMD vs source':  f"{r['mmd_improvement']:+.5f}",
            'Δ MMD (%)':        f"{r['mmd_improvement_pct']:+.1f}%",
        }
        if mode in ('binary', 'multiclass'):
            row['p(male)'] = f"{r['p_male']:.3f}"
            row['95% CI']  = f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]"
            row['n_male / n_female'] = f"{r['n_male']} / {r['n_female']}"
        rows[name] = row
    df = pd.DataFrame(rows).T
    print(f'\n{exp_name} Results:')
    display(df)

    # ── MMD bar chart ──
    names  = list(exp_results.keys())
    mmds   = [exp_results[n]['mmd'] for n in names]
    colors = [METHOD_COLORS.get(n, '#4C72B0') for n in names]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(names, mmds, color=colors, alpha=0.85)
    ax.axhline(source_mmd, color='gray', linestyle='--', linewidth=1.0, label='source baseline')
    for bar, mmd in zip(bars, mmds):
        imp = (source_mmd - mmd) / source_mmd * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                f'{imp:+.1f}%', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('MMD'); ax.set_title(f'{exp_name} — MMD vs target (N={N_EVAL})')
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout(); display(fig); plt.close()

    # ── p(male) CI plot (binary/multiclass only) ──
    if mode in ('binary', 'multiclass'):
        p_hats = [exp_results[n]['p_male'] for n in names]
        ci_lo  = [exp_results[n]['ci_lo']  for n in names]
        ci_hi  = [exp_results[n]['ci_hi']  for n in names]
        target_frac = cfg['groups'][0]['frac']  # man frac

        fig, ax = plt.subplots(figsize=(9, 4))
        for i, (name, p, lo, hi, col) in enumerate(zip(names, p_hats, ci_lo, ci_hi, colors)):
            ax.errorbar(i, p, yerr=[[p-lo], [hi-p]], fmt='o', capsize=6,
                        linewidth=1.5, color=col, markersize=8)
        ax.axhline(target_frac, color='gray', linestyle='--', linewidth=0.8,
                   label=f'target p(male)={target_frac}')
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=15, ha='right')
        ax.set_ylabel('p(male)'); ax.set_title(f'{exp_name} — p(male) 95% CI (N={N_EVAL})')
        ax.set_ylim(0, 1); ax.legend(); ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout(); display(fig); plt.close()

print('\n✅ All experiments done.')

In [ ]:
print(f'Looking in: {base_dir}')
print(f'Exists: {base_dir.exists()}')
if base_dir.exists():
    print('Files found:')
    for f in sorted(base_dir.iterdir()):
        print(f'  {f.name}')
else:
    print('Directory does not exist!')

In [ ]:
## 6. Save Results JSON

# Convert to JSON-serializable
def to_json(obj):
    if isinstance(obj, dict):  return {k: to_json(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [to_json(v) for v in obj]
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

out_path = str(_REPO / 'experiments' / 'eval_all_results.json')
with open(out_path, 'w') as f:
    json.dump(to_json(all_results), f, indent=2)
print(f'Results saved to {out_path}')

# Print summary table across all experiments
print('\n=== SUMMARY ===')
for exp_name, exp_results in all_results.items():
    print(f'\n{exp_name}:')
    for method, r in exp_results.items():
        line = f'  {method:<15} MMD={r["mmd"]:.5f}  Δ={r["mmd_improvement_pct"]:+.1f}%'
        if 'p_male' in r:
            line += f'  p(male)={r["p_male"]:.3f}'
        print(line)